# Phase 6e Wave 1: How does Newton's constant arise from a fermion fluid?

**Stakeholder companion** to `papers/paper39_heat_kernel_expansion/paper_draft.tex`.

This notebook explains the heat-kernel calibration step in plain language and then verifies the closed-form numbers against the Lean-formalized theorems. **Same physics content as the technical notebook**, framed for non-specialist readers.

**The big picture.** This project asks: *can General Relativity be derived as a long-distance description of a fluid of fermions?* The answer is yes — Sakharov (1968) and Adler (1982) showed that integrating out fermions at one loop with a UV cutoff $\Lambda$ generates an Einstein-Hilbert term whose Newton constant is

$$G_N \;=\; \frac{12\pi}{N_f\,\Lambda^2}.$$

Phase 6a.1 of this project formalized that result at the *linearized* level. **Phase 6e Wave 1** — the present module — derives the *same* Newton constant from the *full nonlinear* effective action via Seeley-DeWitt heat-kernel asymptotics, providing a structural cross-check that anchors the rest of Phase 6e.

The match is not numerical; it's a Lean theorem (`G_N_from_a2_eq_G_N_sakharov`) whose proof body invokes the linearized 6a.1 result by name.


## What is a heat kernel, and why do we care?

The "heat kernel" of an operator $\mathit{D}\!\!\!\!/^{\,2}$ is the function $\mathrm{Tr}\,e^{-\tau\,\mathit{D}\!\!\!\!/^{\,2}}$, which describes how a small disturbance "diffuses" through the manifold over the proper time $\tau$. As $\tau \to 0^+$, the trace expands as a series in $\tau$ whose coefficients $a_0, a_2, a_4, \ldots$ are local polynomials in the curvature of the manifold:

- $a_0$ — sets the **vacuum energy** (cosmological-constant scale)
- $a_2$ — sets the **Einstein-Hilbert** Newton constant
- $a_4$ — sets the **higher-curvature** corrections (Riemann², Ricci², Weyl²)

This expansion is the standard tool in quantum field theory in curved spacetime. The coefficients have been known textbook-style since Christensen-Duff 1979 and Gilkey 1995.

**Wave 1 contribution:** Lean theorems pinning each coefficient to the textbook value, plus a structural cross-bridge proving that the $a_2$ coefficient reproduces the Sakharov-Adler $G_N$ exactly.

In [1]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

import math
from src.heat_kernel import (
    seeley_dewitt_coefficients,
    G_N_from_a2,
    a2_calibration_passes,
    a2_calibration_relative_error,
    higher_curvature_dirac_signs,
)

# The fiducial reference: Standard-Model Weyl fermion count per generation
N_f_SM = 15

print(f"Standard-Model fermion count per generation (no ν_R): {N_f_SM}")
print(f"  Quarks (3 colors × 2 chiralities × 2 flavors / gen) + leptons (2 chiralities × 2 flavors / gen)")
print()

# Seeley-DeWitt coefficients at SM N_f
sd = seeley_dewitt_coefficients(N_f_SM)
print(f"Seeley-DeWitt coefficients (per fermion species, in units of (4π)⁻²):")
print(f"  a_0  = +{sd.a0:.4e}      ← cosmological-constant scale")
print(f"  a_2 (R-coef)  = {sd.a2_R_coef:+.4e}   ← Einstein-Hilbert / Newton constant")
print(f"  a_4 (R²)     = {sd.a4_R_sq:+.4e}    ← higher-curvature R² piece")
print(f"  a_4 (Ricci²) = {sd.a4_Ricci_sq:+.4e}    ← higher-curvature Ricci² piece")
print(f"  a_4 (Riem²)  = {sd.a4_Riemann_sq:+.4e}    ← higher-curvature Riemann² piece")

Standard-Model fermion count per generation (no ν_R): 15
  Quarks (3 colors × 2 chiralities × 2 flavors / gen) + leptons (2 chiralities × 2 flavors / gen)

Seeley-DeWitt coefficients (per fermion species, in units of (4π)⁻²):
  a_0  = +3.7995e-01      ← cosmological-constant scale
  a_2 (R-coef)  = -7.9157e-03   ← Einstein-Hilbert / Newton constant
  a_4 (R²)     = -2.1988e-04    ← higher-curvature R² piece
  a_4 (Ricci²) = +3.0783e-04    ← higher-curvature Ricci² piece
  a_4 (Riem²)  = -5.2771e-04    ← higher-curvature Riemann² piece


## What is the Decision Gate E.2 calibration?

The Phase 6e roadmap defines five "decision gates" — falsifiable cross-checks where the formal result either agrees with a published anchor or diagnoses a failure of the underlying assumption.

**Gate E.2** asks: *does the heat-kernel-derived $G_N$ match the Sakharov-Adler-derived $G_N$ at the same $(\Lambda_{UV}, N_f)$?*

If yes, the mean-field approximation underlying ADW emergent gravity is internally consistent at order $\Lambda^2$.

If no, the discrepancy is a load-bearing diagnostic: it pins down where mean-field theory breaks.

In Wave 1, we prove a sharper statement: the match is **exact** at the Sakharov-Adler baseline ($\alpha_{\mathrm{ADW}} = 1$), and the discrepancy is **zero** there. Any deviation from $\alpha_{\mathrm{ADW}} = 1$ in the future Vergeles unitarity calculation will appear as a *quantitatively detectable* mismatch in the heat-kernel calibration.


In [2]:
# Newton's constant at the GUT-scale UV cutoff (10^16 GeV)
Lambda_GUT = 1e16
G_N_HK = G_N_from_a2(Lambda_GUT, N_f_SM)
G_N_closed = 12 * math.pi / (N_f_SM * Lambda_GUT ** 2)

print(f"At Λ_UV = 10¹⁶ GeV (GUT scale), N_f = {N_f_SM}:")
print(f"  Heat-kernel G_N      = {G_N_HK:.4e} GeV⁻²")
print(f"  Sakharov-Adler G_N   = {G_N_closed:.4e} GeV⁻²")
print(f"  Match                = {math.isclose(G_N_HK, G_N_closed, rel_tol=1e-12)}")
print()
print(f"Lean theorem `G_N_from_a2_eq_G_N_sakharov` formalizes this equality.")
print(f"The proof invokes `LinearizedEFE.G_N_sakharov` from Phase 6a.1 by name —")
print(f"a structural cross-bridge protecting against documentation drift.")

At Λ_UV = 10¹⁶ GeV (GUT scale), N_f = 15:
  Heat-kernel G_N      = 2.5133e-32 GeV⁻²
  Sakharov-Adler G_N   = 2.5133e-32 GeV⁻²
  Match                = True

Lean theorem `G_N_from_a2_eq_G_N_sakharov` formalizes this equality.
The proof invokes `LinearizedEFE.G_N_sakharov` from Phase 6a.1 by name —
a structural cross-bridge protecting against documentation drift.


## What does "$\alpha_{\mathrm{ADW}}$ = 1" mean?

$\alpha_{\mathrm{ADW}}$ is a dimensionless coefficient in the Phase 6a.1 emergent Newton constant:

$$G_N^{\mathrm{emerg}}(\Lambda, N_f, \alpha_{\mathrm{ADW}}) = \alpha_{\mathrm{ADW}} \cdot G_N^{\mathrm{Sakharov}}(\Lambda, N_f).$$

At $\alpha_{\mathrm{ADW}} = 1$, the ADW emergent gravity reproduces Sakharov-Adler exactly. The deviation $\alpha_{\mathrm{ADW}} \ne 1$ encodes ADW-specific corrections to the free-fermion limit. Its value awaits the Vergeles unitarity computation (PRD 112, 054509, 2025).

**The Decision Gate E.2 biconditional** — Lean theorem `a2_matches_GNemerg_iff_alpha_ADW_unity` — proves that the heat-kernel $G_N$ matches the linearized $G_N^{\mathrm{emerg}}$ if **and only if** $\alpha_{\mathrm{ADW}} = 1$. Any future deep-research return giving $\alpha_{\mathrm{ADW}} \ne 1$ will *quantitatively diagnose* the mean-field validity boundary at the heat-kernel $a_2$ coefficient.

Below we sweep $\alpha_{\mathrm{ADW}}$ and see how the calibration degrades:

In [3]:
print(f"{'α_ADW':>8}  {'rel error':>11}  {'pass band ±50%':>16}")
for alpha in [0.1, 0.5, 0.9, 1.0, 1.1, 1.5, 2.0, 5.0]:
    err = a2_calibration_relative_error(Lambda_GUT, N_f_SM, alpha)
    pas = a2_calibration_passes(Lambda_GUT, N_f_SM, alpha)
    marker = "  ← Sakharov-Adler baseline (exact match)" if alpha == 1.0 else ""
    print(f"  {alpha:>6.3f}  {err:>11.4f}  {('PASS' if pas else 'FAIL'):>16}{marker}")

   α_ADW    rel error    pass band ±50%
   0.100       9.0000              FAIL
   0.500       1.0000              FAIL
   0.900       0.1111              PASS
   1.000       0.0000              PASS  ← Sakharov-Adler baseline (exact match)
   1.100       0.0909              PASS
   1.500       0.3333              PASS
   2.000       0.5000              PASS
   5.000       0.8000              FAIL


## What about higher curvature ($a_4$)?

Beyond the Einstein-Hilbert term, the next-order correction to the effective action involves three curvature scalars: $R^2$, the Ricci-squared $R_{\mu\nu}R^{\mu\nu}$, and the Riemann-squared $R_{\mu\nu\rho\sigma}R^{\mu\nu\rho\sigma}$. The Christensen-Duff Dirac contribution gives them rational coefficients $-5/2160$, $+7/2160$, $-12/2160$ respectively (per fermion species, before the $(4\pi)^{-2}$ Gaussian factor).

A famous identity in 4D — the **Gauss-Bonnet** combination — is
$$\mathcal{G} = R^2 - 4 R_{\mu\nu}R^{\mu\nu} + R_{\mu\nu\rho\sigma}R^{\mu\nu\rho\sigma}.$$
On a closed 4-manifold, $\int \mathcal{G}\,\sqrt{g}\,d^4 x$ is the Euler characteristic times $32\pi^2$ — i.e., topological. So $\mathcal{G}$ contributes no equations of motion locally; it just labels the manifold's topology.

**Lean check** (`a4_gauss_bonnet_combination`): the linear combination of Christensen-Duff coefficients matching the Gauss-Bonnet contraction evaluates to $-N_f / (48\,(4\pi)^2)$. This is non-zero locally; the topological vanishing arises only after manifold-level integration (deferred to Phase 6f.1 `Curvature.lean`).

In [4]:
# Check the higher-curvature signs and Gauss-Bonnet identity at SM N_f
from src.heat_kernel import gauss_bonnet_combination
signs = higher_curvature_dirac_signs(N_f_SM)
gb = gauss_bonnet_combination(N_f_SM)
gb_closed = -N_f_SM / (48.0 * (4.0 * math.pi) ** 2)
print(f"a_4 coefficient signs (N_f = {N_f_SM}):")
print(f"  R²        : {signs['R_sq']}    (Lean: a4_R_sq_coef_neg)")
print(f"  Ricci²    : {signs['Ricci_sq']}    (Lean: a4_Ricci_sq_coef_pos)")
print(f"  Riemann²  : {signs['Riemann_sq']}    (Lean: a4_Riemann_sq_coef_neg)")
print()
print(f"Gauss-Bonnet local-algebra check:")
print(f"  c_R² − 4 c_Ricci² + c_Riem² = {gb:+.4e}")
print(f"  closed form -N_f/(48 (4π)²) = {gb_closed:+.4e}")
print(f"  match = {math.isclose(gb, gb_closed)}    (Lean: a4_gauss_bonnet_combination)")

a_4 coefficient signs (N_f = 15):
  R²        : neg    (Lean: a4_R_sq_coef_neg)
  Ricci²    : pos    (Lean: a4_Ricci_sq_coef_pos)
  Riemann²  : neg    (Lean: a4_Riemann_sq_coef_neg)

Gauss-Bonnet local-algebra check:
  c_R² − 4 c_Ricci² + c_Riem² = -1.9789e-03
  closed form -N_f/(48 (4π)²) = -1.9789e-03
  match = True    (Lean: a4_gauss_bonnet_combination)


## The figure: visualizing Decision Gate E.2

**Left panel.** The relative error $|G_N^{\mathrm{HK}} - G_N^{\mathrm{lin}}|/G_N^{\mathrm{lin}}$ as a function of $\alpha_{\mathrm{ADW}}$. Exact zero at $\alpha_{\mathrm{ADW}} = 1$. The shaded green band ±50% is the project's mean-field validity policy (matching `GRAV_PARAMS.G_N_MATCH_TOLERANCE`); $\alpha_{\mathrm{ADW}} \in [0.5, 1.5]$ sits inside.

**Right panel.** The Newton constant $G_N(\Lambda_{UV})$ as the UV cutoff is varied across the natural range $[10^{10}, 10^{19}]$ GeV. The CODATA 2018 reference $G_N^{\mathrm{obs}} \approx 6.71 \times 10^{-39}\,\mathrm{GeV}^{-2}$ is the dashed orange line; the curve crosses it near the Planck mass $M_P \approx 1.22 \times 10^{19}$ GeV — the natural scale at which the Sakharov-Adler induced Newton constant matches observation, with no fine-tuning.

In [5]:
# viz-ref: fig_a2_vs_linearized_G_N
from src.core.visualizations import fig_a2_vs_linearized_G_N
fig = fig_a2_vs_linearized_G_N()
fig.show()

## Where does this fit in the bigger picture?

Phase 6e of this project is the heaviest single track: deriving the full nonlinear Einstein equations from the ADW microscopic substrate. Wave 1 — the present module — is the *foundation*: it ships the heat-kernel coefficients that the rest of Phase 6e builds on.

**Downstream waves use Wave 1 as input:**

- Wave 2 (`HigherCurvatureStructure.lean`): uses the $a_4$ coefficients to bound observed higher-curvature deviations against LIGO, pulsar timing, short-range gravity.
- Wave 3 (`NonlinearDiffInvariance.lean`): checks diff-invariance order-by-order using the heat-kernel basis from Wave 1.
- Wave 4 (`NonlinearEFE.lean`): variational derivation of the full nonlinear Einstein equations from the heat-kernel effective action.
- Wave 5 (microscopic↔macroscopic match): expresses $G_N^{\mathrm{emerg}}$, $\Lambda^{\mathrm{emerg}}$, and higher-curvature couplings in microscopic parameters $(\Lambda_{UV}, N_f, G_c)$.
- Wave 6 (`EinsteinCartanExtension.lean`): adds torsion sourced by fermion spin current.

**What "shipping Wave 1" means.** As of 2026-04-27, the Lean module compiles cleanly with 19 substantive theorems, zero `sorry`, zero new axioms beyond the Lean kernel's standard three. A 4-page formalization paper (`paper39_heat_kernel_expansion`) accompanies it. 36 pytest cases verify each Lean theorem against its closed-form Python counterpart.

**What's not yet in Wave 1.** The PDE-level proof of the heat-kernel asymptotic (manifold + spin bundle infrastructure) is a tracked external hypothesis, not a Lean theorem. Mathlib does not yet provide the spin-bundle differential geometry needed; future Mathlib-side work (or a project-side `SpinBundle.lean`) would discharge the tracked hypothesis. This is the same pattern used in Phase 6c.5 RT bounds and Phase 6a.3 BH entropy.

## Lean module quick reference

The file `lean/SKEFTHawking/HeatKernelExpansion.lean` ships **19 substantive theorems** organized by section:

| § | Topic | Theorems |
|---|-------|----------|
| 1 | $(4\pi)^2$ Gaussian normalization | 2 |
| 2 | $a_0$ leading coefficient | 2 |
| 3 | $a_2$ Einstein-Hilbert coefficient | 3 |
| 4 | Tracked structure `DiracHeatKernelAsymptotic` | 3 |
| 5 | Sakharov-Adler calibration | 2 |
| 6 | Quantitative anchors at GUT scale | 2 |
| 7 | $a_4$ higher-curvature basis + Gauss-Bonnet | 4 |
| 8 | Decision Gate E.2 biconditional | 1 |

Build status: `lake build SKEFTHawking.ExtractDeps` clean (190 modules total). Verified axioms: `propext`, `Classical.choice`, `Quot.sound` — the Lean kernel's standard three; no new axioms introduced by Wave 1.

Discipline metric: 2 retroactive theorems caught by post-wave audit, matching the recent best (6c.1=2, 6c.2=2). Trend: 6c.3=12 → 6b.1=5 → 6d.1=6 → 6d.2=4 → 6d.3=1 → 6c.1=2 → 6c.4=3 → 6c.5=3 → 6c.2=2 → **6e.1=2**.